In [2]:
!pip install tokenizers datasets -q



In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import ByteLevel as ByteLevelProcessor
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

In [5]:
tokenizer = Tokenizer(BPE(unk_token=None))

tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.post_processor = ByteLevelProcessor()
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=8000,
    min_frequency=2,
    sepcial_tokens=["<|endoftext|>"],
    show_progress=True,
)

In [6]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

def get_training_corpus():
    for i in range(0, len(dataset), 1000):
        yield dataset[i:i+1000]["text"]

tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)
print(f"Vocab size: {tokenizer.get_vocab_size()}")

vocab = tokenizer.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
print("\n前 20 个 token（base bytes）:")
for token, idx in sorted_vocab[:20]:
    print(f"  {idx:5d}: {repr(token)}")

print("\n最后 20 个 token（最高频合并）:")
for token, idx in sorted_vocab[-20:]:
    print(f"  {idx:5d}: {repr(token)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Vocab size: 8000

前 20 个 token（base bytes）:
      0: '!'
      1: '"'
      2: '#'
      3: '$'
      4: '%'
      5: '&'
      6: "'"
      7: '('
      8: ')'
      9: '*'
     10: '+'
     11: ','
     12: '-'
     13: '.'
     14: '/'
     15: '0'
     16: '1'
     17: '2'
     18: '3'
     19: '4'

最后 20 个 token（最高频合并）:
   7980: 'ĠPortugal'
   7981: 'ĠIsraeli'
   7982: 'Ġloan'
   7983: 'ĠNorway'
   7984: 'criptions'
   7985: 'Ġapparently'
   7986: 'ĠMassachus'
   7987: 'Ġjournalist'
   7988: 'Ġpicture'
   7989: 'Ġchairman'
   7990: 'ĠJacob'
   7991: 'Ġjackrab'
   7992: 'ĠRÃ©union'
   7993: 'Ġcircumstances'
   7994: 'ĠEducation'
   7995: 'ekÄģntav'
   7996: 'ĠBarker'
   7997: 'ĠWiÅĽniowiec'
   7998: 'Ġammunition'
   7999: 'ĠMassachusetts'


In [7]:
test_texts = [
    "Hello, world!",
    "The Transformer architecture revolutionized NLP.",
    "unhappiness",
    "antidisestablishmentarianism",  # 超长低频词
    "12345",                         # 数字
    "你好世界",                       # 中文
]


for text in test_texts:
    output = tokenizer.encode(text)
    print(f"\n原文: {text}")
    print(f"Token IDs: {output.ids}")
    print(f"Tokens: {output.tokens}")
    print(f"Token数量: {len(output.ids)}")

    # 验证可逆性
    decoded = tokenizer.decode(output.ids)
    print(f"还原: {decoded}")
    assert decoded == text, f"还原失败！{decoded} != {text}"

print("\n✅ 所有文本都能完美还原！")


原文: Hello, world!
Token IDs: [39, 415, 78, 11, 1275, 0]
Tokens: ['H', 'ell', 'o', ',', 'Ġworld', '!']
Token数量: 6
还原: Hello, world!

原文: The Transformer architecture revolutionized NLP.
Token IDs: [51, 194, 3743, 649, 198, 4298, 446, 6255, 967, 317, 43, 47, 13]
Tokens: ['T', 'he', 'ĠTrans', 'form', 'er', 'Ġarchitect', 'ure', 'Ġrevolution', 'ized', 'ĠN', 'L', 'P', '.']
Token数量: 13
还原: The Transformer architecture revolutionized NLP.

原文: unhappiness
Token IDs: [301, 71, 1290, 1922]
Tokens: ['un', 'h', 'app', 'iness']
Token数量: 4
还原: unhappiness

原文: antidisestablishmentarianism
Token IDs: [372, 270, 214, 327, 371, 1180, 407, 5481, 1056]
Tokens: ['ant', 'id', 'is', 'est', 'ab', 'lish', 'ment', 'arian', 'ism']
Token数量: 9
还原: antidisestablishmentarianism

原文: 12345
Token IDs: [16, 17, 18, 19, 20]
Tokens: ['1', '2', '3', '4', '5']
Token数量: 5
还原: 12345

原文: 你好世界
Token IDs: [148, 120, 191, 149, 97, 120, 148, 115, 181, 151, 180, 171]
Tokens: ['ä', '½', 'ł', 'å', '¥', '½', 'ä', '¸', 'ĸ', 'ç', 'ķ

In [9]:
import time

def train_tokenizer(vocab_size):
    """训练指定 vocab_size 的 BPE tokenizer"""
    tok = Tokenizer(BPE(unk_token=None))
    tok.pre_tokenizer = ByteLevel(add_prefix_space=False)
    tok.post_processor = ByteLevelProcessor()
    tok.decoder = ByteLevelDecoder()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=["<|endoftext|>"],
        show_progress=False,
    )

    start = time.time()
    tok.train_from_iterator(get_training_corpus(), trainer=trainer)
    train_time = time.time() - start

    return tok, train_time

sizes = [500, 2000, 8000, 32000]
tokenizers_dict = {}

for size in sizes:
    tok, t = train_tokenizer(size)
    tokenizers_dict[size] = tok
    print(f"vocab_size={size:6d}  训练时间: {t:.1f}s")


test_text = """The Transformer architecture, introduced in the paper
"Attention Is All You Need", has fundamentally changed the field of
natural language processing. Models like BERT and GPT demonstrate
the power of pre-training on large corpora."""

print(f"\n测试文本长度: {len(test_text)} chars\n")
print(f"{'Vocab Size':>12} | {'Token数':>8} | {'压缩率':>8} | 示例 tokens")
print("-" * 80)

for size in sizes:
    tok = tokenizers_dict[size]
    encoded = tok.encode(test_text)
    n_tokens = len(encoded.ids)
    ratio = len(test_text) / n_tokens  # chars per token
    sample = encoded.tokens[:5]
    print(f"{size:>12,} | {n_tokens:>8} | {ratio:>7.1f}x | {sample}")

# 关键指标：chars per token（越高 = 效率越高）

vocab_size=   500  训练时间: 3.7s
vocab_size=  2000  训练时间: 3.8s
vocab_size=  8000  训练时间: 5.1s
vocab_size= 32000  训练时间: 4.4s

测试文本长度: 234 chars

  Vocab Size |   Token数 |      压缩率 | 示例 tokens
--------------------------------------------------------------------------------
         500 |      126 |     1.9x | ['T', 'he', 'ĠT', 'ran', 's']
       2,000 |       92 |     2.5x | ['T', 'he', 'ĠT', 'rans', 'form']
       8,000 |       71 |     3.3x | ['T', 'he', 'ĠTrans', 'form', 'er']
      32,000 |       59 |     4.0x | ['T', 'he', 'ĠTrans', 'form', 'er']


In [11]:
from transformers import AutoTokenizer

# 加载主流模型的 tokenizer
tokenizers_compare = {
    "BERT (WordPiece, 30k)": AutoTokenizer.from_pretrained("bert-base-uncased"),
    "GPT-2 (BPE, 50k)": AutoTokenizer.from_pretrained("gpt2"),
    "LLaMA-2 (SP+BPE, 32k)": AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0"),
}
# 如果 LLaMA 需要认证，改用 TinyLlama:
# tokenizers_compare["LLaMA-style (SP+BPE, 32k)"] = AutoTokenizer.from_pretrained(
#     "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# )

test_texts_multi = [
    "Hello, how are you?",
    "The quick brown fox jumps over the lazy dog.",
    "unhappiness",
    "你好世界",  # 中文
    "123456789",  # 数字
    "https://www.example.com/path?query=value",  # URL
]

for text in test_texts_multi:
    print(f"\n{'='*60}")
    print(f"原文: {text}")
    print(f"{'='*60}")
    for name, tok in tokenizers_compare.items():
        tokens = tok.tokenize(text)
        ids = tok.encode(text, add_special_tokens=False)
        print(f"{name:>30}: {len(ids):2d} tokens → {tokens}")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]


原文: Hello, how are you?
         BERT (WordPiece, 30k):  6 tokens → ['hello', ',', 'how', 'are', 'you', '?']
              GPT-2 (BPE, 50k):  6 tokens → ['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']
         LLaMA-2 (SP+BPE, 32k):  6 tokens → ['▁Hello', ',', '▁how', '▁are', '▁you', '?']

原文: The quick brown fox jumps over the lazy dog.
         BERT (WordPiece, 30k): 10 tokens → ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
              GPT-2 (BPE, 50k): 10 tokens → ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.']
         LLaMA-2 (SP+BPE, 32k): 12 tokens → ['▁The', '▁quick', '▁brown', '▁fo', 'x', '▁j', 'umps', '▁over', '▁the', '▁lazy', '▁dog', '.']

原文: unhappiness
         BERT (WordPiece, 30k):  4 tokens → ['un', '##ha', '##pp', '##iness']
              GPT-2 (BPE, 50k):  3 tokens → ['un', 'h', 'appiness']
         LLaMA-2 (SP+BPE, 32k):  3 tokens → ['▁unh', 'app', 'iness']

原文: 你好世界
         BERT (WordPiece, 30k):  